# Сборка датасета для модели сутяжности клиента

Пайплайн вынесен в пакет `querulus.dataset`. Тетрадка только настраивает окружение и вызывает `run_pipeline`.

## Настройка

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import sys
from pathlib import Path

# Корень репозитория и src/ — как в integration/tests/__init__.py
_here = Path.cwd().resolve()
PROJECT_ROOT = next(
    p for p in (_here, *_here.parents) if (p / "pyproject.toml").exists()
)
SRC = PROJECT_ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

In [ ]:
import logging

import pandas as pd

# без scientific notation в display (3.45e+04 -> 34545.00)
pd.options.display.float_format = "{:,.2f}".format

from querulus.dataset import run_pipeline
from querulus.dataset.io import setup_notebook_logging

setup_notebook_logging(logging.INFO)

# True: подгрузить готовый финальный датасет; False: пересобрать пайплайн
LOAD_FINAL_DATASET = True
FINAL_DATASET_PATH = PROJECT_ROOT / "data" / "processed" / "df_final_3.parquet"
# LOAD_FINAL_DATASET = True
# FINAL_DATASET_PATH = Path("/home/jovyan/old_home/Litigant/data/processed/df_final_3.parquet")

# None=авто; True=старый parquet; False=новый (пайплайн или icnl-датасет)
LOAD_LEGACY_DATASET = None

# False: parquet из data/raw (или SQL при отсутствии); True: всегда SQL
USE_SQL = False
SAVE_CHECKPOINT = True

# Три стека таргетов (обучение сравнивает все):
#   legacy:     TARGET_2 + TARGET_3_SEV
#   new:        TARGET_FREQ + TARGET_SEV              (иски + претензии)
#   new_claims: TARGET_FREQ_CLAIMS + TARGET_SEV_CLAIMS (только иски)
# Признаки: "selected" = selected_features.py | "mvp" = полный MVP-пул
# Для отбора фич: FEATURES_SOURCE="mvp" + SELECT_FEATURES=True (только стек new)
FEATURES_SOURCE = "mvp"  # "selected" | "mvp"
SELECT_FEATURES = True  # RecursiveByShapValues: freq+sev по 30 фич (только new)
# Для EDA / диагностики / графиков — какой стек показывать детально
VIEW_STACK = "new"  # "legacy" | "new" | "new_claims"

from querulus.training.triple_stack import TARGET_STACKS

STACK_TARGETS = {name: (freq, sev) for name, freq, sev in TARGET_STACKS}
FREQUENCY_TARGET, SEVERITY_TARGET = STACK_TARGETS[VIEW_STACK]
print(f"VIEW_STACK={VIEW_STACK}: {FREQUENCY_TARGET} + {SEVERITY_TARGET}")


## Пайплайн

In [ ]:
from querulus.dataset.steps.targets import ensure_claims_targets

if LOAD_FINAL_DATASET:
    if not FINAL_DATASET_PATH.exists():
        raise FileNotFoundError(
            f"Финальный датасет не найден: {FINAL_DATASET_PATH}. "
            "Поставьте LOAD_FINAL_DATASET=False, чтобы собрать его заново."
        )
    df = pd.read_parquet(FINAL_DATASET_PATH)
    print(f"LOAD final dataset: {FINAL_DATASET_PATH} shape={df.shape}")
else:
    df = run_pipeline(use_sql=USE_SQL, save_checkpoint=SAVE_CHECKPOINT)

df = ensure_claims_targets(df)
FREQUENCY_TARGET, SEVERITY_TARGET = STACK_TARGETS[VIEW_STACK]
print(f"VIEW_STACK={VIEW_STACK}: {FREQUENCY_TARGET} + {SEVERITY_TARGET}, columns ok="
      f"{FREQUENCY_TARGET in df.columns and SEVERITY_TARGET in df.columns}")


## Проверка результата

In [ ]:
from querulus.training.triple_stack import TARGET_STACKS

# Проверка распределений по всем стекам (не зависит от VIEW_STACK)
for stack_name, freq_col, sev_col in TARGET_STACKS:
    print(f"=== {stack_name}: {freq_col} / {sev_col} ===")
    if freq_col not in df.columns:
        print(f"нет колонки {freq_col} — пересоберите датасет или ensure_claims_targets")
        continue
    display(df[freq_col].value_counts(dropna=False))
    if sev_col not in df.columns:
        print(f"нет колонки {sev_col}")
        continue
    display(df[sev_col].describe())


In [ ]:
# Проверка: freq==0 при sev>0 (для new и new_claims)
for freq_col, sev_col in (("TARGET_FREQ", "TARGET_SEV"), ("TARGET_FREQ_CLAIMS", "TARGET_SEV_CLAIMS")):
    if freq_col not in df.columns or sev_col not in df.columns:
        print(f"skip {freq_col}/{sev_col}: колонок нет")
        continue
    mask = (df[freq_col].fillna(0) == 0) & (df[sev_col].fillna(0) > 0)
    print(f"{freq_col}==0 & {sev_col}>0: {int(mask.sum())} / {len(df)}")
    cols = [c for c in ["INCIDENT_NUMBER", "LOSS_NUMBER", freq_col, sev_col] if c in df.columns]
    display(df.loc[mask, cols].head(20))


## Сверка таргетов

1. **Litigant (old) vs Querulus:** `TARGET_2`, `TARGET_3_SEV` + топ-20.
2. **На Querulus — 6 таргетов:** legacy / new / new_claims (пары old↔new, old↔claims, new↔claims) + топ-20.


In [ ]:
from querulus.training.target_compare import (
    default_litigant_dataset_path,
    run_target_comparison_suite,
)

# litigant_path = Path("/home/jovyan/old_home/Litigant/data/processed/df_final_3.parquet")
suite = run_target_comparison_suite(
    df,
    litigant_path=default_litigant_dataset_path(),
    top_n=20,
)

if suite.litigant is not None:
    print("=== 1. Litigant vs Querulus (TARGET_2, TARGET_3_SEV) ===")
    display(suite.litigant.report)
    print("=== 2. Топ-20: классификация TARGET_2 (old vs new) ===")
    display(suite.litigant.top_classification)
    print("=== 2. Топ-20: регрессия TARGET_3_SEV (old vs new) ===")
    display(suite.litigant.top_regression)
else:
    print(f"Litigant parquet не найден — блок 1–2 пропущен ({default_litigant_dataset_path()})")

print("=== 3. Статистика: legacy / new / new_claims (6 таргетов) ===")
display(suite.old_vs_new.report)
print("=== 4. Топ-20: классификация ===")
display(suite.old_vs_new.top_classification)
print("=== 4. Топ-20: регрессия ===")
display(suite.old_vs_new.top_regression)


## EDA признаков

`research_continous` / `research_feature` по MVP-признакам (до обучения, как в `model_learn.py`).

In [ ]:
from querulus.training.config import TrainingConfig
from querulus.training.plots import run_mvp_frequency_eda

EDA_CONFIG = TrainingConfig(
    frequency_target=FREQUENCY_TARGET,
    severity_target=SEVERITY_TARGET,
    features_source=FEATURES_SOURCE,
)
if "expos" not in df.columns:
    df["expos"] = 1

run_mvp_frequency_eda(df, EDA_CONFIG)


## Обучение трёх стеков

Обучаем `legacy` / `new` / `new_claims`. Признаки: `FEATURES_SOURCE` (`selected` | `mvp`).
Feature select (freq+sev, по 30) — только стек **new**, если `SELECT_FEATURES=True` и source=`mvp`.
`severity_range` снят: severity на всём диапазоне таргета.


In [ ]:
import importlib

importlib.invalidate_caches()

from IPython.display import Markdown, display

from querulus.training.config import TrainingConfig
from querulus.training.pipeline import format_features_table, format_training_summary
from querulus.training.triple_stack import run_triple_stack

TRAINING_CONFIG = TrainingConfig(
    features_source=FEATURES_SOURCE,
    frequency_select_features=SELECT_FEATURES,
    severity_select_features=SELECT_FEATURES,
    frequency_num_features_to_select=30,
    severity_num_features_to_select=30,
    frequency_calibration_enabled=False,
    severity_range=None,
)
triple = run_triple_stack(
    df,
    TRAINING_CONFIG,
    split="test",
    loaded_from_checkpoint=LOAD_FINAL_DATASET,
    legacy_dataset=LOAD_LEGACY_DATASET,
    run_fin_effect=True,
)

# Для диагностики / графиков — выбранный VIEW_STACK
training = triple.trainings[VIEW_STACK]
FREQUENCY_TARGET, SEVERITY_TARGET = STACK_TARGETS[VIEW_STACK]

display(Markdown("### Metrics summary (legacy / new / new_claims)"))
display(triple.metrics_summary)

display(Markdown(f"### Training summary ({VIEW_STACK})"))
display(format_training_summary(training.summary))

display(Markdown(f"### Frequency features ({VIEW_STACK})"))
display(format_features_table(training.frequency_features, training.frequency_categorical_features))

display(Markdown(f"### Severity features ({VIEW_STACK})"))
display(format_features_table(training.severity_features, training.severity_categorical_features))

display(Markdown(f"### Frequency importance ({VIEW_STACK})"))
display(training.frequency_importance.head(30))
display(Markdown(f"### Severity importance ({VIEW_STACK})"))
display(training.severity_importance.head(30))


## Дрейф фич и severity-хвост (VIEW_STACK)

- PSI / L1 train vs test по отобранным фичам + топ importance.
- Ошибка severity по квантилям true (бьёт ли хвост).
- A/B raw vs `log1p` severity на том же сплите.


In [ ]:
from querulus.training.drift import feature_drift_report, monthly_target_drift
from querulus.training.severity_diagnostics import (
    compare_severity_log1p,
    severity_error_by_quantile,
)

# Календарный дрейф фич (frequency + severity, VIEW_STACK)
drift_features = list(
    dict.fromkeys(training.frequency_features + training.severity_features)
)
drift_cats = list(
    dict.fromkeys(
        training.frequency_categorical_features + training.severity_categorical_features
    )
)
imp = pd.concat(
    [training.frequency_importance, training.severity_importance],
    ignore_index=True,
)
if not imp.empty:
    imp = (
        imp.groupby("feature", as_index=False)["importance"]
        .max()
        .sort_values("importance", ascending=False)
    )

drift_report = feature_drift_report(
    df,
    drift_features,
    TRAINING_CONFIG,
    categorical_features=drift_cats,
    importance=imp,
    top_importance=30,
)
display(Markdown(f"### Feature drift train vs test ({VIEW_STACK})"))
display(drift_report.head(40))

display(Markdown(f"### Monthly target mean ({FREQUENCY_TARGET})"))
display(monthly_target_drift(df, FREQUENCY_TARGET, TRAINING_CONFIG))
display(Markdown(f"### Monthly target mean ({SEVERITY_TARGET})"))
display(monthly_target_drift(df, SEVERITY_TARGET, TRAINING_CONFIG))

# Хвост ошибки severity + сравнение log1p
assert training.severity_split is not None
y_true_sev = training.severity_split.y_test
y_pred_sev = training.severity_model.predict(
    training.severity_split.x_test[training.severity_features]
)
sev_q = severity_error_by_quantile(y_true_sev, y_pred_sev)
display(Markdown(f"### Severity error by true quantile ({VIEW_STACK})"))
display(sev_q)

log1p_cmp = compare_severity_log1p(training, TRAINING_CONFIG)
display(Markdown("### Severity raw vs log1p (global)"))
display(log1p_cmp.summary.pivot(index="metric", columns="model", values="value"))
display(Markdown("### Severity error by quantile — raw"))
display(log1p_cmp.quantile_raw)
display(Markdown("### Severity error by quantile — log1p"))
display(log1p_cmp.quantile_log1p)


## Диагностика модели

Распределение вероятностей классификации и ModelDiagnostics (после обучения).

In [ ]:
from querulus.training.plots import run_model_diagnostics_visualizations

# Диагностика стека VIEW_STACK (training = triple.trainings[VIEW_STACK])
print(f"diagnostics for VIEW_STACK={VIEW_STACK}: {FREQUENCY_TARGET} + {SEVERITY_TARGET}")
run_model_diagnostics_visualizations(df, training, TRAINING_CONFIG)


## Финансовый эффект (три стека)

Сводная таблица: threshold / fact / model / net по `legacy`, `new`, `new_claims`.
Детальная сводка квадрантов — для `VIEW_STACK`.


In [ ]:
from querulus.fin_effect import (
    create_summary_table,
    export_analytics_excel,
    export_summary_excel,
    print_best_threshold_report,
    resolve_fin_effect_config,
)

display(Markdown("### Fin-effect summary (3 stacks)"))
display(triple.fin_effect_summary)

FIN_EFFECT_CONFIG = resolve_fin_effect_config(
    df,
    frequency_target=FREQUENCY_TARGET,
    severity_target=SEVERITY_TARGET,
    loaded_from_checkpoint=LOAD_FINAL_DATASET,
    legacy_dataset=LOAD_LEGACY_DATASET,
)
fin_effect = triple.fin_effects[VIEW_STACK]
print(f"VIEW_STACK={VIEW_STACK} fact_mode={FIN_EFFECT_CONFIG.fact_mode}")
print_best_threshold_report(fin_effect)

FIN_EFFECT_DIR = PROJECT_ROOT / "data" / "processed" / "fin_effect"
FIN_EFFECT_DIR.mkdir(parents=True, exist_ok=True)

summary_table = create_summary_table(fin_effect.frame, FIN_EFFECT_CONFIG)
display(summary_table.style.format("{:,.0f}", subset=summary_table.columns[3:]))

export_summary_excel(summary_table, FIN_EFFECT_DIR / f"summary_table_{VIEW_STACK}.xlsx")
export_analytics_excel(
    fin_effect.frame, FIN_EFFECT_DIR / f"Аналитика_{VIEW_STACK}.xlsx", config=FIN_EFFECT_CONFIG
)

triple.fin_effect_summary.to_excel(FIN_EFFECT_DIR / "fin_effect_triple_summary.xlsx", index=False)
triple.metrics_summary.to_excel(FIN_EFFECT_DIR / "metrics_triple_summary.xlsx", index=False)
print(f"Экспорт: {FIN_EFFECT_DIR}")


In [ ]:
from querulus.fin_effect import (
    plot_confusion_matrix,
    plot_cost_confusion_heatmaps,
    plot_positive_cases_by_month,
    plot_precision_recall_vs_threshold,
    plot_severity_fact_vs_pred_binned,
    plot_target_monthly_share,
)

split = training.frequency_split
frame = fin_effect.frame
y_test = split.y_test.loc[frame.index]
x_test = split.x_test.loc[frame.index, training.frequency_features]

business_threshold = fin_effect.best_threshold
plot_confusion_matrix(
    frame[FIN_EFFECT_CONFIG.frequency_target_column],
    frame["pred_freq"],
    title=f"Confusion Matrix {FREQUENCY_TARGET} (threshold={business_threshold:.2f})",
)
plot_precision_recall_vs_threshold(training.frequency_model, x_test, y_test)
plot_cost_confusion_heatmaps(frame, y_test, frame["pred_freq"])
plot_severity_fact_vs_pred_binned(
    frame,
    frame[SEVERITY_TARGET],
    frame["pred_sev"],
    mask_actual_claim=(y_test == 1),
    config=FIN_EFFECT_CONFIG,
)
plot_target_monthly_share(df, config=FIN_EFFECT_CONFIG)
plot_positive_cases_by_month(df, config=FIN_EFFECT_CONFIG)

## База факта и взносы

Три базы: **legacy** (ПСР), **new** (`TARGET_FREQ_AMOUNT` = claims+pret), **new_claims** (`TARGET_FREQ_CLAIMS_AMOUNT`, без претензий).
Взносы считаются отдельно под каждую базу (триггер ФУ только при base > 0).


In [ ]:
from IPython.display import Markdown, display

from querulus.fin_effect import (
    compare_fact_bases,
    compare_premiums,
    compare_severity_targets,
)

pd.options.display.float_format = "{:,.2f}".format

print("=== Severity facts: TARGET_3_SEV vs TARGET_SEV ===")
sev_fact = compare_severity_targets(df)
display(sev_fact)
top = sev_fact.attrs.get("top_mismatches")
if top is not None and not top.empty:
    display(Markdown("Топ расхождений severity (факт)"))
    display(top)

print("=== Severity facts: TARGET_SEV vs TARGET_SEV_CLAIMS ===")
sev_claims = compare_severity_targets(df, legacy_col="TARGET_SEV", new_col="TARGET_SEV_CLAIMS")
display(sev_claims)

print("=== Денежная база: legacy / new / new_claims ===")
display(compare_fact_bases(df))

print("=== Взносы: отдельно для legacy / new / new_claims ===")
display(compare_premiums(df))

print("=== Fin-effect summary (повтор из обучения) ===")
display(triple.fin_effect_summary)
